# ExLightNet gpu_v2 on Colab (B2, B5, B4 neural)

1. Runtime -> Change runtime type -> **T4 GPU** (write down the GPU name the next cells print; every later session must use the SAME GPU type).
2. Left Files panel: upload `hosted_source.zip` and `prepared.zip` (zip of the `prepared` folder: manifest.json + split_42..51.npz) into `/content`.
3. Run cells 1 -> 2 (pre-flight: must end with READY) -> 3 (install) -> 4 (the real run; keep this tab open) -> 5 (download).
4. If the session dies: reconnect, upload the partly finished zip as `/content/resume_gpu_v2.zip`, rerun cells 1 -> 4. Finished runs are skipped.

In [ ]:
# Cell 1: unpack the uploads, set the deterministic environment, look at the GPU.
import os, sys, subprocess, shutil, zipfile
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_name] = "1"

CONTENT = Path("/content")
for _zip, _dest in (("hosted_source.zip", "code"), ("prepared.zip", "prepared")):
    _z = CONTENT / _zip
    if _z.is_file() and not (CONTENT / _dest).exists():
        with zipfile.ZipFile(_z) as _zf:
            _zf.extractall(CONTENT / _dest)
        print("unpacked", _zip, "->", CONTENT / _dest)

if shutil.which("nvidia-smi"):
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                          capture_output=True, text=True)
    print("GPU:", _smi.stdout.strip() or _smi.stderr.strip())
else:
    print("GPU: NONE -> Runtime -> Change runtime type -> T4 GPU, reconnect, then rerun from cell 1")
print("in /content:", sorted(p.name for p in CONTENT.iterdir()))

In [ ]:
# ExLightNet Colab PRE-FLIGHT. Read-only check: trains nothing, writes only temporary files.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be set before CUDA is touched
import sys, json, hashlib, shutil, subprocess, tempfile, time, warnings
import importlib.metadata as md
from pathlib import Path

INPUT_BASE = Path("/content")
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)
EXPECTED_MANIFEST_SHA256 = "95eb221ecbd74f172d978abd58d3f5c2863ac9708288446a331b6fce980b7b35"
SPLITS = range(42, 52)
rows, state = [], {"needs_patch": False}


def check(name, ok, detail="", fatal=True):
    rows.append((name, ok, fatal))
    tag = "PASS" if ok else ("FAIL" if fatal else "WARN")
    print(f"[{tag}] {name}" + (f" :: {detail}" if detail else ""), flush=True)
    return ok


def guard(name, fn, fatal=True):
    try:
        return check(name, True, fn() or "")
    except Exception as exc:
        return check(name, False, f"{type(exc).__name__}: {str(exc)[:300]}", fatal)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# 1. Environment ---------------------------------------------------------------
check("Python >= 3.11", sys.version_info >= (3, 11), sys.version.split()[0])
free_gb = shutil.disk_usage(WORK if WORK.exists() else "/").free / 1e9
check("Free disk in the working directory >= 2 GB", free_gb >= 2, f"{free_gb:.1f} GB", fatal=False)
for pkg in ("numpy", "torch", "scikit-learn", "setuptools"):
    try:
        check(f"package {pkg}", True, md.version(pkg))
    except md.PackageNotFoundError:
        check(f"package {pkg}", False, "missing")
xgb = None
for pkg in ("xgboost", "xgboost-cpu"):
    try:
        xgb = f"{pkg} {md.version(pkg)}"
        break
    except md.PackageNotFoundError:
        pass
check("package xgboost", xgb is not None, str(xgb))

# 2. Locate the two attached datasets ----------------------------------------
SOURCE = PREPARED = None
if INPUT_BASE.exists():
    for root, dirs, files in os.walk(INPUT_BASE):
        dirs[:] = [d for d in dirs if d not in {"site", "src_copy", "out", "work", "sample_data"}]
        r = Path(root)
        if len(r.relative_to(INPUT_BASE).parts) > 5:
            dirs[:] = []
            continue
        if SOURCE is None and "pyproject.toml" in files and (r / "scripts" / "kaggle_campaign.py").is_file() and (r / "src" / "exlightnet").is_dir():
            SOURCE = r
        if PREPARED is None and "manifest.json" in files and (r / "split_42.npz").is_file():
            PREPARED = r
check("code dataset found (pyproject.toml + scripts/ + src/exlightnet)", SOURCE is not None, str(SOURCE))
check("prepared dataset found (manifest.json + split_42.npz)", PREPARED is not None, str(PREPARED))
if SOURCE is None or PREPARED is None:
    top = sorted(p.name for p in INPUT_BASE.iterdir()) if INPUT_BASE.exists() else []
    raise RuntimeError(f"Upload hosted_source.zip and prepared.zip to /content (Files panel), rerun cell 1, then this cell. Found under {INPUT_BASE}: {top}")

# 3. Prepared arrays match their manifest -------------------------------------
manifest_path = PREPARED / "manifest.json"
check("manifest.json is the exact file Codex delivered", sha256(manifest_path) == EXPECTED_MANIFEST_SHA256,
      sha256(manifest_path)[:16] + "...", fatal=False)
entries = json.loads(manifest_path.read_text()).get("splits", {})
for s in SPLITS:
    p = PREPARED / f"split_{s}.npz"
    e = entries.get(str(s))
    ok = p.is_file() and e is not None and sha256(p) == e.get("sha256")
    check(f"split_{s}.npz matches manifest sha256", ok, f"{p.stat().st_size / 1e6:.1f} MB" if p.is_file() else "missing")

# 4. Offline install, exactly as the runner cell does it ---------------------
def pip_install(src, target):
    return subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation",
                           "--upgrade", "--target", str(target), str(src)], capture_output=True, text=True)


tmp = Path(tempfile.mkdtemp(prefix="preflight_", dir=str(WORK) if WORK.exists() else None))
site = tmp / "site"
r = pip_install(SOURCE, site)
if r.returncode == 0:
    check("offline install from the dataset path (as the runner does)", True)
else:
    state["needs_patch"] = True
    check("offline install from the dataset path (as the runner does)", False,
          (r.stdout + r.stderr).strip()[-350:] + "  -> runner setup cell needs the patch printed below", fatal=False)
    copy = tmp / "src_copy"
    shutil.copytree(SOURCE, copy)
    r2 = pip_install(copy, site)
    check("offline install from a writable COPY of the dataset", r2.returncode == 0,
          "ok" if r2.returncode == 0 else (r2.stdout + r2.stderr).strip()[-350:])
if not (site / "exlightnet").is_dir():
    raise RuntimeError("Package could not be installed offline; see the messages above.")

env = dict(os.environ, PYTHONPATH=str(site))
cli = subprocess.run([sys.executable, str(SOURCE / "scripts" / "kaggle_campaign.py"), "--help"],
                     capture_output=True, text=True, env=env)
check("runner CLI starts and offers --only", cli.returncode == 0 and "--only" in cli.stdout, cli.stderr.strip()[-300:])
sys.path.insert(0, str(site))


def plans():
    from exlightnet.campaign import study_runs
    counts = {s: len(study_runs(s)) for s in ("B1", "B2", "B3")}
    seeds = [r["seed"] for r in study_runs("B1")]
    assert counts == {"B1": 330, "B2": 90, "B3": 180}, counts
    assert seeds == sorted(seeds), "B1 is not seed-major"
    return f"runs {counts}; B1 is seed-major"


guard("study plans load (B1=330, B2=90, B3=180, seed-major)", plans)

# 5. Load split 42 with the project's own loader and manifest check -----------
data = {}


def load_split():
    from exlightnet.prepared import load_prepared
    from exlightnet.campaign import verify_prepared_manifest
    prep = load_prepared(PREPARED / "split_42.npz")
    verify_prepared_manifest(PREPARED, 42, prep["metadata"])
    shape = tuple(prep["arrays"]["x_train"].shape)
    assert shape == (86181, 40), shape
    data["prep"] = prep
    return f"x_train {shape}"


guard("split_42 loads and passes the code's own manifest check", load_split)

# 6. GPU and a real deterministic ExLightNet training step ----------------------
def gpu_step():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA device. Runtime -> Change runtime type -> a GPU, then reconnect and rerun from cell 1.")
    name, cap = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
    print(f"      GPU: {name}, capability {cap}, visible GPUs {torch.cuda.device_count()}, torch {torch.__version__}, CUDA {torch.version.cuda}")
    t = torch.randn(64, 64, device="cuda")
    float((t @ t).sum().item())  # fails with 'no kernel image' if torch does not support this GPU
    from exlightnet.baselines import build_model
    from exlightnet.device import configure_device
    from exlightnet.losses import FocalLoss, class_alpha
    configure_device("cuda")
    prep = data["prep"]
    arrays = prep["arrays"]
    torch.manual_seed(42)
    model = build_model("exlightnet", 12, 42).to("cuda").train()
    loss_fn = FocalLoss(class_alpha(arrays["y12_train"], prep["metadata"]["frozen_alpha"]["multiclass"])).to("cuda")
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    xb = torch.as_tensor(arrays["x_train"][:512], dtype=torch.float32, device="cuda")
    yb = torch.as_tensor(arrays["y12_train"][:512], dtype=torch.long, device="cuda")

    def step():
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        return loss

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for _ in range(3):
            step()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(20):
            loss = step()
        torch.cuda.synchronize()
        per_step = (time.perf_counter() - t0) / 20
    assert bool(torch.isfinite(loss)), "non-finite loss"
    print("      deterministic-mode warnings (informational):", sorted({str(w.message)[:150] for w in caught}) or "none")
    return f"{per_step * 1000:.1f} ms/step -> roughly {per_step * 169:.1f} s/epoch (rough; the probe gives the official number)"


guard("GPU works and one real ExLightNet training step runs with the deterministic flags", gpu_step)

# 7. Clean up and summarise -----------------------------------------------------
sys.path.remove(str(site))
for name in [k for k in sys.modules if k.startswith("exlightnet")]:
    del sys.modules[name]
shutil.rmtree(tmp, ignore_errors=True)

print("=" * 72)
failed = [n for n, ok, fatal in rows if not ok and fatal]
print("Paths found (the next cells use them automatically):")
print(f'  SOURCE   = Path("{SOURCE}")')
print(f'  PREPARED = Path("{PREPARED}")')
if state["needs_patch"]:
    print("\nNote: direct install failed; the next cell installs from a writable copy, so this is expected.")
print("=" * 72)
if failed:
    print("NOT READY. Failed checks:", *failed, sep="\n  - ")
    raise RuntimeError("Pre-flight failed; fix the items above before running anything else.")
print("READY" + (" after applying the patch above." if state["needs_patch"] else ". You can run the probe now."))

In [ ]:
# Cell 3: install the package from a writable copy (offline-style, no dependencies).
SITE = Path("/content/site")
COPY = Path("/content/src_copy")
OUT = Path("/content/out")
OUT.mkdir(exist_ok=True)

shutil.rmtree(COPY, ignore_errors=True)
shutil.copytree(SOURCE, COPY)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation",
                "--upgrade", "--target", str(SITE), str(COPY)], check=True)
os.environ["PYTHONPATH"] = str(SITE)
print("installed; code at", SOURCE, "| prepared at", PREPARED, "| output ->", OUT)

In [ ]:
# Cell 4: the real run. Keep this browser tab open. Order: probe, B2 (neural only), B5, B4.
RESUME = sorted(Path("/content").glob("resume_*.zip"))
print("resuming from:", [p.name for p in RESUME] or "nothing (fresh start)")


def run_study(study, only="all", device="cuda", budget_hours=6):
    command = [sys.executable, str(SOURCE / "scripts/kaggle_campaign.py"),
               "--prepared-dir", str(PREPARED), "--output", str(OUT),
               "--study", study, "--device", device, "--only", only, "--campaign", "gpu_v2",
               "--time-budget-hours", str(budget_hours), "--zip-every", "5"]
    for _z in RESUME:
        command += ["--resume-zip", str(_z)]
    proc = subprocess.Popen(command, env=os.environ.copy(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for _line in proc.stdout:
        print(_line, end="", flush=True)
    rc = proc.wait()
    assert rc == 0, f"{study} failed with exit code {rc}"


run_study("probe")
run_study("B2", only="neural")
run_study("B5")
run_study("B4")
print("ALL DONE. Now run cell 5 to download the zip.")

In [ ]:
# Cell 5: count the records and download the zip. Expected after a complete run: 111 record.json files.
import zipfile
from google.colab import files

_zip = Path("/content/out/gpu_v2.zip")
with zipfile.ZipFile(_zip) as _zf:
    _n = sum(1 for _name in _zf.namelist() if _name.endswith("record.json"))
print(f"{_zip.name}: {_zip.stat().st_size / 1e6:.1f} MB, {_n} record.json files (expected 111 when everything finished)")
files.download(str(_zip))